# 🧵 DeepLure Saree Design Recognition
## Color-Invariant Visual Design Retrieval via Self-Supervised Metric Learning

---

### 🎯 Objective
Build a **color-invariant visual saree design recognition and retrieval system**.
The system analyzes the fundamental visual characteristics of saree patterns — including **motifs, shapes, repeated geometric structures, border layouts, weave texture, and spatial composition** — independent of color palette variations.

### 🛡️ Non-Negotiable Identity Policy
1. **Zero Filename Bias**: Image filenames, file paths, and hashes are NEVER used as design identities.
2. **Category as Metadata**: `Banarasi`, `Bandhani`, `Ikat`, and `Pichwai` are coarse weaving categories, NOT individual design IDs.
3. **Color-Invariance**: Color is decoupled from identity via dual-view contrastive learning (pulling together two color-perturbed views of the same weave pattern).
4. **Unit Hypersphere Embedding**: Each saree is mapped to an L2-normalized 512-D visual vector ($||z||_2 = 1.0$), enabling fast cosine similarity retrieval.

## 1. Environment & Hardware Detection

In [ ]:
import os
import sys
import time
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from sklearn.manifold import TSNE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Set seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# Detect Compute Hardware
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('=' * 50)
print('SYSTEM & HARDWARE CONFIGURATION')
print('=' * 50)
print(f'Python Version:  {sys.version.split()[0]}')
print(f'PyTorch Version: {torch.__version__}')
print(f'Compute Device:  {device}')
if torch.cuda.is_available():
    print(f'GPU Device:      {torch.cuda.get_device_name(0)}')
    print(f'VRAM Total:      {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
print('=' * 50)

## 2. Dataset Discovery & Visual Inventory Audit

In [ ]:
# Candidate dataset paths (auto-detects Kaggle input or local path)
CANDIDATE_PATHS = [
    Path('/kaggle/input/datasets/div456/indian-saree-patterns'),
    Path('/kaggle/input/indian-saree-patterns'),
    Path('./kaggle'),
    Path('../kaggle')
]

DATA_ROOT = None
for p in CANDIDATE_PATHS:
    if p.exists() and (p / 'train').exists():
        DATA_ROOT = p
        break

if DATA_ROOT is None:
    raise FileNotFoundError('Could not locate Indian Saree Patterns dataset path.')

print(f'Located Dataset Root: {DATA_ROOT}')

# Scan all valid images across splits
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
records = []

for split in ['train', 'valid', 'test']:
    split_dir = DATA_ROOT / split
    if not split_dir.exists():
        continue
    for img_path in split_dir.rglob('*'):
        if img_path.is_file() and img_path.suffix.lower() in VALID_EXTS:
            records.append({
                'path': str(img_path),
                'split': split,
                'category': img_path.parent.name
            })

df_meta = pd.DataFrame(records)

print('\n' + '=' * 50)
print('DATASET AUDIT INVENTORY')
print('=' * 50)
print(f'Total Valid Images: {len(df_meta)}')
print('\nSplit Breakdown:')
print(df_meta['split'].value_counts())
print('\nCategory Distribution (Coarse Metadata):')
print(df_meta['category'].value_counts())
print('=' * 50)

In [ ]:
# Visual Inspection of Sample Saree Patterns
categories = sorted(df_meta['category'].unique())
fig, axes = plt.subplots(len(categories), 4, figsize=(14, 10))

for r, cat in enumerate(categories):
    cat_samples = df_meta[df_meta['category'] == cat]['path'].tolist()[:4]
    for c, img_p in enumerate(cat_samples):
        ax = axes[r, c]
        with Image.open(img_p) as img:
            ax.imshow(img.convert('RGB'))
        if c == 0:
            ax.set_ylabel(cat, fontsize=12, fontweight='bold')
        ax.axis('off')

plt.suptitle('Sample Visual Saree Motifs by Category', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Color-Invariant Augmentations & Two-View Contrastive Dataset

To teach the network that **saree design identity is invariant to color palette variations**, we create a two-view dataset where:
- Every original image yields **two differently augmented views** ($x_i, x_i^+$).
- Both views maintain the **exact same weave motifs, shapes, and spatial layout**.
- The color palette is perturbed via aggressive hue shift, saturation changes, contrast jitter, and grayscale conversions.

In [ ]:
# Realistic Color-Invariant Augmentation Pipeline
contrastive_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(
        brightness=0.35,
        contrast=0.30,
        saturation=0.40,
        hue=0.08
    ),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class TwoViewSareeDataset(Dataset):
    """Yields two distinct augmented views of the same saree image."""
    def __init__(self, records_list, transform):
        self.records = records_list
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        path = self.records[idx]['path']
        with Image.open(path) as raw:
            img = raw.convert('RGB')
        v1 = self.transform(img)
        v2 = self.transform(img)
        return v1, v2, self.records[idx]['category']

class SingleViewSareeDataset(Dataset):
    """Yields single image tensor + metadata for inference and validation."""
    def __init__(self, records_list, transform):
        self.records = records_list
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        item = self.records[idx]
        with Image.open(item['path']) as raw:
            img = raw.convert('RGB')
        tensor_img = self.transform(img)
        return tensor_img, item['path'], item['category']

train_records = df_meta[df_meta['split'] == 'train'].to_dict('records')
valid_records = df_meta[df_meta['split'] == 'valid'].to_dict('records')
test_records  = df_meta[df_meta['split'] == 'test'].to_dict('records')

train_contrast_ds = TwoViewSareeDataset(train_records, contrastive_transform)
valid_contrast_ds = TwoViewSareeDataset(valid_records, contrastive_transform)

train_loader = DataLoader(train_contrast_ds, batch_size=32, shuffle=True, drop_last=True)
valid_loader = DataLoader(valid_contrast_ds, batch_size=32, shuffle=False, drop_last=False)

print(f'Train Contrastive Batches: {len(train_loader)} | Valid Batches: {len(valid_loader)}')

In [ ]:
# Demonstration of Two-View Color-Invariance Transformation on a Single Saree
sample_path = train_records[0]['path']
with Image.open(sample_path) as raw:
    sample_img = raw.convert('RGB')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(sample_img)
axes[0].set_title('Original Saree Image', fontweight='bold')
axes[0].axis('off')

# Denormalize helper for plotting tensor
def unnormalize(t):
    inv = t.clone()
    for c, (m, s) in enumerate(zip([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])):
        inv[c] = inv[c] * s + m
    return torch.clamp(inv.permute(1, 2, 0), 0, 1).numpy()

v1 = contrastive_transform(sample_img)
v2 = contrastive_transform(sample_img)

axes[1].imshow(unnormalize(v1))
axes[1].set_title('Augmented View 1\n(Color/Scale Perturbed)', fontweight='bold')
axes[1].axis('off')

axes[2].imshow(unnormalize(v2))
axes[2].set_title('Augmented View 2\n(Color/Scale Perturbed)', fontweight='bold')
axes[2].axis('off')

plt.suptitle('Positive Pair for Contrastive Learning: Same Weave Motifs, Distinct Colorways', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Visual Embedding Model Architecture (ResNet-50 + 512-D L2 Head)

In [ ]:
class SareeEmbeddingModel(nn.Module):
    """
    Deep Visual Saree Design Embedding Model.
    Maps raw saree images -> 512-D L2-normalized representations on unit hypersphere S^511.
    """
    def __init__(self, embedding_dim=512, pretrained=True):
        super().__init__()
        weights = ResNet50_Weights.DEFAULT if pretrained else None
        self.backbone = resnet50(weights=weights)
        
        # Access 2048-dim global average pooled features
        feature_dim = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        
        # 512-D Projection Head
        self.embedding_head = nn.Sequential(
            nn.Linear(feature_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2),
            nn.Linear(1024, embedding_dim)
        )
        self.embedding_dim = embedding_dim

    def forward(self, x):
        features = self.backbone(x)
        proj = self.embedding_head(features)
        embedding = F.normalize(proj, p=2, dim=1) # L2 normalization
        return embedding

    @torch.no_grad()
    def extract_embedding(self, x):
        self.eval()
        return self.forward(x)

model = SareeEmbeddingModel(embedding_dim=512, pretrained=True).to(device)

# Forward pass test
dummy_in = torch.randn(4, 3, 224, 224).to(device)
dummy_out = model(dummy_in)

backbone_params = sum(p.numel() for p in model.backbone.parameters())
head_params = sum(p.numel() for p in model.embedding_head.parameters())

print('=' * 50)
print('MODEL ARCHITECTURE & SPECIFICATIONS')
print('=' * 50)
print(f'Backbone:               Pretrained ResNet50 (ImageNet)')
print(f'Backbone Parameters:    {backbone_params:,}')
print(f'Embedding Head Params:  {head_params:,}')
print(f'Total Parameters:       {backbone_params + head_params:,}')
print(f'Output Embedding Dim:   {dummy_out.shape[1]}-D')
print(f'Unit Norm Check (L2):   {torch.norm(dummy_out, dim=1).tolist()}')
print('=' * 50)

## 5. Contrastive Metric Loss & Self-Supervised Training Loop

In [ ]:
class ContrastiveInfoNCELoss(nn.Module):
    """
    Normalized Temperature-scaled Cross Entropy Loss (NT-Xent / SimCLR).
    Pulls together two augmented views of the same saree while pushing apart all other sarees.
    """
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, z_i, z_j):
        batch_size = z_i.size(0)
        dev = z_i.device
        
        z = torch.cat([z_i, z_j], dim=0) # [2B, D]
        sim_matrix = torch.matmul(z, z.T) / self.temperature # [2B, 2B]
        
        # Mask diagonal
        diag_mask = torch.eye(2 * batch_size, dtype=torch.bool, device=dev)
        sim_matrix.masked_fill_(diag_mask, -1e9)
        
        targets = torch.cat([
            torch.arange(batch_size, 2 * batch_size, device=dev),
            torch.arange(0, batch_size, device=dev)
        ])
        
        return F.cross_entropy(sim_matrix, targets)

# Optimizer and Loss Setup
criterion = ContrastiveInfoNCELoss(temperature=0.07).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=12)

In [ ]:
# Execute Training Loop
EPOCHS = 12
history = []
best_val_loss = float('inf')

os.makedirs('./checkpoints', exist_ok=True)
best_checkpoint_path = './checkpoints/best_model.pt'

print('=' * 60)
print('STARTING COLOR-INVARIANT CONTRASTIVE TRAINING')
print('=' * 60)

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    train_loss = 0.0
    t0 = time.time()
    
    for v1, v2, _ in train_loader:
        v1, v2 = v1.to(device), v2.to(device)
        optimizer.zero_grad()
        z1 = model(v1)
        z2 = model(v2)
        loss = criterion(z1, z2)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()
        
    scheduler.step()
    avg_train_loss = train_loss / len(train_loader)
    
    # --- Validation ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for v1, v2, _ in valid_loader:
            v1, v2 = v1.to(device), v2.to(device)
            z1 = model(v1)
            z2 = model(v2)
            loss = criterion(z1, z2)
            val_loss += loss.item()
            
    avg_val_loss = val_loss / len(valid_loader)
    elapsed = time.time() - t0
    
    print(f'Epoch {epoch:02d}/{EPOCHS:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Time: {elapsed:.1f}s')
    
    history.append({
        'epoch': epoch,
        'train_loss': avg_train_loss,
        'val_loss': avg_val_loss,
        'lr': optimizer.param_groups[0]['lr']
    })
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_loss': avg_val_loss,
            'embedding_dim': 512
        }, best_checkpoint_path)
        print(f'  ★ Checkpoint saved to {best_checkpoint_path}')

# Save training history
pd.DataFrame(history).to_csv('./training_history.csv', index=False)
print('\nTraining complete. History saved to ./training_history.csv')

## 6. Training Curves & Color-Invariance Validation

In [ ]:
# Load best checkpoint
ckpt = torch.load(best_checkpoint_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

# Plot Training & Validation Loss Curves
hist_df = pd.DataFrame(history)
plt.figure(figsize=(9, 4.5))
plt.plot(hist_df['epoch'], hist_df['train_loss'], label='Train Contrastive Loss', color='royalblue', linewidth=2)
plt.plot(hist_df['epoch'], hist_df['val_loss'], label='Val Contrastive Loss', color='crimson', linewidth=2, linestyle='--')
plt.title('Self-Supervised Contrastive Training Curves', fontsize=12, fontweight='bold')
plt.xlabel('Epoch', fontsize=11)
plt.ylabel('InfoNCE Loss', fontsize=11)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Calculate Color Invariance Score (Cosine similarity of two distinct colorways of same saree)
invariance_scores = []
with torch.no_grad():
    for v1, v2, _ in valid_loader:
        v1, v2 = v1.to(device), v2.to(device)
        z1 = model.extract_embedding(v1)
        z2 = model.extract_embedding(v2)
        sim = (z1 * z2).sum(dim=1) # Dot product on unit sphere
        invariance_scores.extend(sim.cpu().tolist())

mean_invariance = float(np.mean(invariance_scores))
print('=' * 50)
print('REPRESENTATION VALIDATION METRIC')
print('=' * 50)
print(f'Mean Color-Invariance Alignment Score: {mean_invariance:.4f}')
print('Explanation: Cosine similarity between two differently colored/viewpoint')
print('versions of the same saree pattern. Range: [-1.0, 1.0], Higher is better.')
print('=' * 50)

## 7. Gallery Index Construction

In [ ]:
# Build Gallery Index from Training Images
train_single_ds = SingleViewSareeDataset(train_records, inference_transform)
gallery_loader = DataLoader(train_single_ds, batch_size=32, shuffle=False)

gallery_embeddings = []
gallery_paths = []
gallery_categories = []

with torch.no_grad():
    for imgs, paths, cats in tqdm(gallery_loader, desc='Building Gallery Index'):
        imgs = imgs.to(device)
        emb = model.extract_embedding(imgs)
        gallery_embeddings.append(emb.cpu())
        gallery_paths.extend(paths)
        gallery_categories.extend(cats)

gallery_embeddings = torch.cat(gallery_embeddings, dim=0) # [N, 512]
torch.save(gallery_embeddings, './gallery_embeddings.pt')

print(f'Gallery Index Built: {gallery_embeddings.shape[0]} items indexed in 512-D space.')

## 8. Query Retrieval & Top-K Similarity Visualization

In [ ]:
def retrieve_top_k(query_path, top_k=5):
    """
    Visual retrieval for any query image using normalized cosine similarity.
    """
    with Image.open(query_path) as raw:
        query_img = raw.convert('RGB')
    
    tensor_q = inference_transform(query_img).unsqueeze(0).to(device)
    with torch.no_grad():
        query_emb = model.extract_embedding(tensor_q).cpu() # [1, 512]
        
    # Cosine similarities (dot product on unit sphere)
    similarities = torch.matmul(gallery_embeddings, query_emb.squeeze(0))
    top_scores, top_indices = torch.topk(similarities, k=top_k)
    
    results = []
    for rank, (idx, score) in enumerate(zip(top_indices.tolist(), top_scores.tolist()), start=1):
        results.append({
            'rank': rank,
            'score': float(score),
            'path': gallery_paths[idx],
            'category': gallery_categories[idx]
        })
    return query_img, results

# Test query on a random test-split saree image
sample_query_path = test_records[0]['path']
q_img, top_matches = retrieve_top_k(sample_query_path, top_k=5)

# Display Top-5 Visual Matches
fig, axes = plt.subplots(1, 6, figsize=(18, 3.8))
axes[0].imshow(q_img)
axes[0].set_title('QUERY SAREE\n(Test Image)', fontweight='bold', color='navy')
axes[0].axis('off')

for i, match in enumerate(top_matches, start=1):
    ax = axes[i]
    with Image.open(match['path']) as m_img:
        ax.imshow(m_img.convert('RGB'))
    ax.set_title(f'Rank #{match["rank"]}\nSim: {match["score"]:.4f}\n({match["category"]})', fontsize=10)
    ax.axis('off')

plt.suptitle('DeepLure Top-5 Visual Design Retrieval Demonstration', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. 2D t-SNE Visualization of Visual Embeddings

In [ ]:
# Extract features for validation set
valid_single_ds = SingleViewSareeDataset(valid_records, inference_transform)
val_loader_single = DataLoader(valid_single_ds, batch_size=32, shuffle=False)

val_embs, val_cats = [], []
with torch.no_grad():
    for imgs, _, cats in val_loader_single:
        imgs = imgs.to(device)
        e = model.extract_embedding(imgs)
        val_embs.append(e.cpu().numpy())
        val_cats.extend(cats)

val_embs = np.concatenate(val_embs, axis=0)

# Run t-SNE
tsne = TSNE(n_components=2, perplexity=15, random_state=42, init='pca')
embs_2d = tsne.fit_transform(val_embs)

tsne_df = pd.DataFrame({
    't-SNE Dim 1': embs_2d[:, 0],
    't-SNE Dim 2': embs_2d[:, 1],
    'Category': val_cats
})

plt.figure(figsize=(9, 6))
sns.scatterplot(data=tsne_df, x='t-SNE Dim 1', y='t-SNE Dim 2', hue='Category', palette='tab10', s=70, alpha=0.9)
plt.title('t-SNE Projection of 512-D Color-Invariant Visual Embeddings', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Category-Level Diagnostic Evaluation (Metadata Probe)

> **Evaluation Note**: The provided Kaggle dataset contains 4 broad category annotations (`Banarasi`, `Bandhani`, `Ikat`, `Pichwai`), but does not provide individual fine-grained design IDs. The k-NN probe below evaluates whether the self-supervised visual features naturally capture the distinct weaving styles without ever training on category labels.

In [ ]:
# Diagnostic k-NN probe on Category Metadata
X_train = gallery_embeddings.numpy()
y_train = gallery_categories
X_val = val_embs
y_val = val_cats

knn1 = KNeighborsClassifier(n_neighbors=1, metric='cosine')
knn1.fit(X_train, y_train)
knn1_acc = float(accuracy_score(y_val, knn1.predict(X_val)))

knn5 = KNeighborsClassifier(n_neighbors=5, metric='cosine')
knn5.fit(X_train, y_train)
knn5_acc = float(accuracy_score(y_val, knn5.predict(X_val)))

print('=' * 50)
print('CATEGORY-LEVEL DIAGNOSTIC PROBE')
print('=' * 50)
print(f'1-NN Category Accuracy (Cosine): {knn1_acc * 100:.2f}%')
print(f'5-NN Category Accuracy (Cosine): {knn5_acc * 100:.2f}%')
print('=' * 50)

## 11. Efficiency Benchmark & Final Deliverable Results

In [ ]:
# Measure exact inference and retrieval latencies
dummy_q = torch.randn(1, 3, 224, 224).to(device)

# Warmup
for _ in range(10):
    _ = model.extract_embedding(dummy_q)

# Inference Latency
latencies = []
for _ in range(100):
    t_start = time.perf_counter()
    _ = model.extract_embedding(dummy_q)
    latencies.append((time.perf_counter() - t_start) * 1000)
mean_inference_ms = float(np.mean(latencies))

# Gallery Retrieval Latency (Dot Product search across N gallery items)
g_tensor = gallery_embeddings.to(device)
q_emb_cuda = torch.randn(512, device=device)

retrieval_latencies = []
for _ in range(100):
    t_start = time.perf_counter()
    _ = torch.matmul(g_tensor, q_emb_cuda)
    retrieval_latencies.append((time.perf_counter() - t_start) * 1000)
mean_retrieval_ms = float(np.mean(retrieval_latencies))

results_dict = {
    'model': 'ResNet50 + 512-D L2 Projection Head',
    'total_parameters': backbone_params + head_params,
    'embedding_dimension': 512,
    'embedding_size_bytes_per_image': 512 * 4, # 2 KB
    'mean_color_invariance_score': round(mean_invariance, 4),
    'diagnostic_category_knn1_acc': round(knn1_acc, 4),
    'single_image_inference_ms': round(mean_inference_ms, 2),
    'gallery_search_latency_ms': round(mean_retrieval_ms, 4)
}

with open('./evaluation_results.json', 'w') as f:
    json.dump(results_dict, f, indent=2)

print('=' * 60)
print('FINAL DEEPLURE MEASURED RESULTS (Deliverables Summary)')
print('=' * 60)
print(f'Total Model Parameters:          {results_dict["total_parameters"]:,}')
print(f'Embedding Dimensionality:        {results_dict["embedding_dimension"]}-D (2 KB / image)')
print(f'Color Invariance Alignment:      {results_dict["mean_color_invariance_score"]:.4f}')
print(f'Diagnostic Category Probe Acc:   {results_dict["diagnostic_category_knn1_acc"] * 100:.2f}%')
print(f'Single Image Inference Latency:  {results_dict["single_image_inference_ms"]:.2f} ms')
print(f'Gallery Retrieval Search Time:   {results_dict["gallery_search_latency_ms"]:.4f} ms')
print('=' * 60)
print('Execution completed successfully. Checkpoints and reports saved.')